<a href="https://colab.research.google.com/github/atefehjamshidnia/sql-customer-risk-analysis/blob/main/Lending_Club_Risk_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adarshsng/lending-club-loan-data-csv")

print("Path to dataset files:", path)

100%|██████████| 339M/339M [00:13<00:00, 25.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/adarshsng/lending-club-loan-data-csv/versions/1


In [ ]:
import pandas as pd
import sqlite3
import glob

# 1. Define the exact path
path = "/root/.cache/kagglehub/datasets/adarshsng/lending-club-loan-data-csv"
csv_file = glob.glob(path + "/**/*.csv", recursive=True)[0]
print("CSV file located at:", csv_file)

# 2. Create database connection
conn = sqlite3.connect("risk_analysis.db")

print("Transferring data to SQL database in chunks... (This will take a moment)")

# 3. Read and write in chunks to save RAM (50,000 rows at a time)
chunk_size = 50000
for i, chunk in enumerate(pd.read_csv(csv_file, chunksize=chunk_size, low_memory=False)):
    if i == 0:
        # First chunk creates the table
        chunk.to_sql("loan_data", conn, if_exists="replace", index=False)
    else:
        # Next chunks append to the existing table
        chunk.to_sql("loan_data", conn, if_exists="append", index=False)
    print(f"Chunk {i+1} transferred...")

print("✅ Database setup complete!")

# 4. Test query
test_query = "SELECT COUNT(*) AS Total_Rows FROM loan_data;"
result = pd.read_sql(test_query, conn)
display(result)

CSV file located at: /root/.cache/kagglehub/datasets/adarshsng/lending-club-loan-data-csv/versions/1/loan.csv
Transferring data to SQL database in chunks... (This will take a moment)
Chunk 1 transferred...
Chunk 2 transferred...
Chunk 3 transferred...
Chunk 4 transferred...
Chunk 5 transferred...
Chunk 6 transferred...
Chunk 7 transferred...
Chunk 8 transferred...
Chunk 9 transferred...
Chunk 10 transferred...
Chunk 11 transferred...
Chunk 12 transferred...
Chunk 13 transferred...
Chunk 14 transferred...
Chunk 15 transferred...
Chunk 16 transferred...
Chunk 17 transferred...
Chunk 18 transferred...
Chunk 19 transferred...
Chunk 20 transferred...
Chunk 21 transferred...
Chunk 22 transferred...
Chunk 23 transferred...
Chunk 24 transferred...
Chunk 25 transferred...
Chunk 26 transferred...
Chunk 27 transferred...
Chunk 28 transferred...
Chunk 29 transferred...
Chunk 30 transferred...
Chunk 31 transferred...
Chunk 32 transferred...
Chunk 33 transferred...
Chunk 34 transferred...
Chunk 35 t

,Total_Rows
0,2260668


In [2]:
# Query 2: Average loan amount and income by loan status
query_avg = """
SELECT
    loan_status,
    COUNT(*) as Total_Loans,
    ROUND(AVG(loan_amnt), 2) as Avg_Loan_Amount,
    ROUND(AVG(annual_inc), 2) as Avg_Annual_Income
FROM
    loan_data
GROUP BY
    loan_status;
"""

df_avg = pd.read_sql(query_avg, conn)
display(df_avg)

,loan_status,Total_Loans,Avg_Loan_Amount,Avg_Annual_Income
0,Charged Off,261655,15548.98,70327.78
1,Current,919695,15884.65,80611.45
2,Default,31,15800.00,66074.19
3,Does not meet the credit policy. Status:Charge...,761,9527.23,69525.92
4,Does not meet the credit policy. Status:Fully ...,1988,8853.23,72145.42
5,Fully Paid,1041952,14132.49,77623.33
6,In Grace Period,8952,17696.06,81109.84
7,Late (16-30 days),3737,16983.24,79393.48
8,Late (31-120 days),21897,16715.30,76469.19


In [3]:
import pandas as pd
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect("risk_analysis.db")

# Query 3: Create a comprehensive aggregated report for the Excel dashboard
excel_query = """
SELECT
    loan_status,
    grade,
    term,
    COUNT(*) AS Total_Loans,
    ROUND(AVG(loan_amnt), 0) AS Avg_Loan_Amount,
    ROUND(AVG(int_rate), 2) AS Avg_Interest_Rate,
    ROUND(AVG(annual_inc), 0) AS Avg_Annual_Income,
    ROUND(AVG(dti), 2) AS Avg_DTI
FROM
    loan_data
WHERE
    loan_status IN ('Fully Paid', 'Charged Off', 'Current')
GROUP BY
    loan_status, grade, term
ORDER BY
    grade, loan_status;
"""

print("Generating the aggregated report for Excel...")
df_excel = pd.read_sql(excel_query, conn)

# Save the resulting DataFrame as a CSV file for Excel
file_name = "Risk_Analysis_Dashboard_Data.csv"
df_excel.to_csv(file_name, index=False)

print(f"✅ File '{file_name}' has been created successfully!")

# Display the first 10 rows to verify the data
display(df_excel.head(10))

Generating the aggregated report for Excel...
✅ File 'Risk_Analysis_Dashboard_Data.csv' has been created successfully!


,loan_status,grade,term,Total_Loans,Avg_Loan_Amount,Avg_Interest_Rate,Avg_Annual_Income,Avg_DTI
0,Charged Off,A,36 months,13136,13407.0,7.37,79610.0,17.18
1,Charged Off,A,60 months,630,19602.0,7.89,110278.0,16.00
2,Current,A,36 months,186755,14681.0,7.00,89536.0,16.95
3,Current,A,60 months,17680,22967.0,7.48,106951.0,16.92
4,Fully Paid,A,36 months,207094,13750.0,7.09,88968.0,15.52
5,Fully Paid,A,60 months,5383,19162.0,7.87,103503.0,14.51
6,Charged Off,B,36 months,42757,12219.0,10.80,70023.0,18.43
7,Charged Off,B,60 months,8320,20333.0,10.78,89072.0,18.17
8,Current,B,36 months,194550,12612.0,10.64,76402.0,18.59
9,Current,B,60 months,80905,22244.0,10.70,95823.0,19.33


In [4]:
from google.colab import files

# Download the file directly to your computer
files.download("Risk_Analysis_Dashboard_Data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>